In [1]:

import numpy as np
import pandas as pd

from coin_flip_with_riskless_asset_model import CoinFlipWithRisklessAssetModel

In [ ]:
def simulate_running_empirical_growth_rates(rng: np.random.Generator, params: CoinFlipWithRisklessAssetModel, weights_vector: np.ndarray, num_simulations: int, size: int) -> np.ndarray:
    '''
    Simulates the running empirical growth rates for num_simulations simulations of size periods each, given the parameters and weights vector. Returns a 2D array of shape (num_simulations, size), where each row is a simulation of the running empirical growth rates.
    '''
    array_of_riskless_gross_returns = np.ones(size)*params.r
    results = []
    for _ in range(num_simulations):
        random_outcomes = rng.choice([params.gamma_tails, params.gamma_heads], size=size, p=params.probabilities_vector)
        matrix_of_outcomes = np.column_stack((array_of_riskless_gross_returns, random_outcomes))
        results.append(return_running_empirical_growth_rates(np.dot(matrix_of_outcomes, weights_vector)))
    return np.array(results)

In [4]:
def augment_df_with_summary_stats(df: pd.DataFrame, threshold: float=0.0) -> pd.DataFrame:
    '''
    Assuming df has shape (num_simulations, size), this function returns a new DataFrame with shape (num_simulations, size + 6), where the first size columns are the same as df, and the last 6 columns are the fraction of simulations that are positive, the fraction of simulations that are negative, the net diffusion index, the mean, the median, and the 95% confidence interval for each period. The index of the DataFrame is the period number, starting from 1.
    '''
    df_augmented = df.copy()
    df_augmented['Fraction Above'] = (df > threshold).mean(axis=1)
    df_augmented['Fraction Below'] = (df < threshold).mean(axis=1)
    df_augmented['Net Diffusion Index'] = df_augmented['Fraction Above'] - df_augmented['Fraction Below']
    df_augmented['Mean'] = df.mean(axis=1)
    df_augmented['Median'] = df.median(axis=1)
    df_augmented['95% CI Lower'] = df.apply(lambda x: np.percentile(x, 2.5), axis=1)
    df_augmented['95% CI Upper'] = df.apply(lambda x: np.percentile(x, 97.5), axis=1)

    return df_augmented

In [5]:
def return_empirical_growth_rates_dataframe_for_plotting(running_empirical_growth_rates: np.ndarray) -> pd.DataFrame:
    df = pd.DataFrame(running_empirical_growth_rates, columns=np.arange(1, running_empirical_growth_rates.shape[1] + 1))
    return augment_df_with_summary_stats(df)

def return_wealth_over_time_dataframe_for_plotting(running_empirical_growth_rates: np.ndarray) -> pd.DataFrame:
    df_growth_rates = pd.DataFrame(running_empirical_growth_rates, columns=np.arange(1, running_empirical_growth_rates.shape[1] + 1))
    df_wealth_relative = np.exp(df_growth_rates*np.arange(1, df_growth_rates.shape[1] + 1))
    return augment_df_with_summary_stats(df_wealth_relative, threshold=1.0)  # type: ignore

def return_wealth_over_time_dataframe_for_plotting_2(running_empirical_growth_rates: np.ndarray) -> pd.DataFrame:
    wealth_over_time_array = np.exp(running_empirical_growth_rates*np.arange(1, running_empirical_growth_rates.shape[1] + 1))
    return augment_df_with_summary_stats(wealth_over_time_array, threshold=1.0)  # type: ignore

In [ ]:




# GENERATE DATA FOR THE PLOT OF THE ARITHMETIC VS GEOMETRIC RETURN AS A FUNCTION OF ALLOCATION TO THE RISKY ASSET

def generate_arithmetic_vs_geometric_data(params: CoinFlipWithRisklessAssetModel) -> pd.DataFrame:

    output = []
    for f in np.linspace(0, 1, 1001):
        weights_vector = np.array([1 - f, f])
        arithmetic_gross_return = params.return_arithmetic_portfolio_gross_return(weights_vector) # E[Y] = weights_vector . E[X]
        geometric_gross_return = params.return_geometric_portfolio_gross_return(weights_vector) # exp(E[log(Y)])
        growth_rate = params.return_expected_log_portfolio_gross_return(weights_vector) # E[log(Y)]
        growth_rate_ceiling = np.log(params.return_arithmetic_portfolio_gross_return(weights_vector)) # log(E[Y]). This follows from Jensen's inequality.

        output.append((f, arithmetic_gross_return, geometric_gross_return, growth_rate, growth_rate_ceiling))
    df_arith_geo = pd.DataFrame(output, columns=['f', 'Arithmetic Gross Return', 'Geometric Gross Return', 'Growth Rate', 'Growth Rate Ceiling'])
    return df_arith_geo

In [ ]:

params = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, p=0.5, alpha=1.25, r=0.97)



In [ ]:
f_star = opt_result_for_CRP.x

weights_for_risky_asset = np.array([0.0, 1.0])
weights_for_optimal_CRP = np.array([1 - f_star, f_star])
num_simulations = 200
size = 15 # number of periods to simulate

In [ ]:
running_empirical_growth_rates = simulate_running_empirical_growth_rates(rng, params, weights_for_optimal_CRP, num_simulations, size)

df_risky_asset_for_wealth = return_wealth_over_time_dataframe_for_plotting(running_empirical_growth_rates)
df_optimal_CRP_for_wealth = return_wealth_over_time_dataframe_for_plotting(running_empirical_growth_rates)

fig_risky_asset_for_wealth = create_wealth_over_time_plot(df_risky_asset_for_wealth, title="Simulating the Risky Asset")
fig_optimal_CRP_for_wealth = create_wealth_over_time_plot(df_optimal_CRP_for_wealth, title="Simulating the Optimal CRP")

In [ ]:
df_risky_asset_for_growth_rate = return_empirical_growth_rates_dataframe_for_plotting(running_empirical_growth_rates)
df_optimal_CRP_for_growth_rate = return_empirical_growth_rates_dataframe_for_plotting(running_empirical_growth_rates)

fig_risky_asset_for_growth_rate = create_empirical_growth_rates_plot(df_risky_asset_for_growth_rate, weights_for_risky_asset, params, title="Simulating the Risky Asset")
fig_optimal_CRP_for_growth_rate = create_empirical_growth_rates_plot(df_optimal_CRP_for_growth_rate, weights_for_optimal_CRP, params, title="Simulating the Optimal CRP")

In [ ]:
df_arith_geo = generate_arithmetic_vs_geometric_data(params)
fig_arith_geo = generate_arithmetic_vs_geometric_plot(df_arith_geo, opt_result_for_CRP)

In [ ]:
# Set the parameters for the model

params = CoinFlipWithRisklessAssetModel(gamma_heads=2.0, p=0.5, alpha=1.25, r=0.97)
